In [9]:
import pandas as pd
import json
import io

def get_valid_json_lines(file_path):
    """Filters a JSONL file and yields only valid JSON lines."""
    with open(file_path, "r", encoding='utf-8') as f:
        for line in f:
            try:
                json.loads(line)
                yield line
            except json.JSONDecodeError:
                continue

file_path = "C:/Users/edcastr/OneDrive - Microsoft/Documents/NSU/Spring 2026/TainingDataset2.jsonl"
valid_lines = "".join(get_valid_json_lines(file_path)) # Combine valid lines into a single string
# Read the combined string with lines=True
df = pd.read_json(io.StringIO(valid_lines), lines=True)
df

,Question,Answer
0,Which JavaScript library is primarily used to ...,React
1,Which library is often used for creating singl...,Vue.js
2,Which JavaScript framework is known for its tw...,Angular
3,Which library provides a simple way to animate...,AOS
4,Which library is popular for creating responsi...,Slick Carousel
...,...,...
2702,What is a lambda function in Python?,"I am sorry, I do not have an answer for your q..."
2703,How do you serialize an object in C#?,"I am sorry, I do not have an answer for your q..."
2704,How do you use smart pointers in C++?,"I am sorry, I do not have an answer for your q..."
2705,What is the apply function family in R?,"I am sorry, I do not have an answer for your q..."


In [1]:
import pandas as pd

#df = pd.read_json('TainingDatasetGPT5_gemma-2b-it_answers2.jsonl', lines=True)
#df = pd.read_json('ValidationDataset2GPT5_gemma-2b-it_answers_edcastro_v8.jsonl', lines=True)
#df = pd.read_json('ValidationDataset2GPT5_tinyLlama_answers_OOB.jsonl', lines=True)
#df = pd.read_json('ValidationDataset2GPT5_DeepSeek-R1-Distill-Qwen_answers_edcastro_v9.jsonl', lines=True)
df = pd.read_json('ValidationDataset2GPT5_Phi-3-mini-4k-instruct-LoRA_answers_edcastro_v9.jsonl', lines=True)
df

,Question,Answer,FT,OOB
0,"In JavaScript development, which library is wi...",React,React,"React is widely used for building declarative,..."
1,Which JavaScript library makes it easier to ha...,Socket.io,Socket.io,"For handling real-time, bidirectional communic..."
2,"In JavaScript, which library is commonly used ...",marked,marked,The commonly used library in JavaScript to par...
3,What JavaScript library lets you define compon...,styled-components,emotion,One JavaScript library that allows you to defi...
4,"In a React single-page application, which Java...",React Router,React Router,The commonly used JavaScript library for handl...
...,...,...,...,...
494,How can you import modules in Python?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...",Importing modules in Python is a fundamental a...
495,How would you implement a thread in C#?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...",Implementing a thread in C# involves creating ...
496,How do you allocate memory dynamically in C++?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...",Allocating memory dynamically in C++ is a fund...
497,How do you find the mean of a vector in R?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...","To find the mean of a numeric vector in R, you..."


In [ ]:
from openai import OpenAI
import json
from pprint import pprint

def Evaluate(Question,ExpectedAnswer,LLMAnswer):
    client = OpenAI(api_key="your key here")
    messages=[
        {
          "role": "developer",
          "content": [
            {
              "type": "input_text",
              "text": "Evaluate whether the LLM's output satisfactorily answers a question about JavaScript libraries by comparing it to the provided expected answer. The LLM's response is considered correct (score: 1) if it provides at least one relevant library name that matches any part of the expected answer—even if the expected answer lists multiple libraries and only one is provided. Respond with 0 only if there is no overlap or the answer is irrelevant. Do not penalize concise or partially complete but relevant responses.\n\nYou will receive:\n- question: [user's question about JavaScript libraries]\n- expected_answer: [expected library/libraries, comma-separated if multiple]\n- llm_output: [LLM's answer]\n\nFollow these steps:\n1. Review the question for context.  \n2. Compare llm_output against expected_answer:\n    - If any correct library in expected_answer appears in llm_output, classify as correct (1).\n    - If the expected answer lists multiple libraries (e.g., \"React, Vue, Angular\") and llm_output has just one (e.g., \"React\"), score 1.\n    - Ignore case and minor spelling variations.  \n    - Generic or irrelevant answers not naming a library get 0. \n    - The library must be exact, for example if the expected answer is \"React\" but the llm_output is \"Reacttacular\" then score 0.\n- If the question is expected to not be answered, if the llm_output is like \"I am sorry I cannot answer\" should score 1.\n3. Output only JSON in this format: {\"score\": 1} or {\"score\": 0}. No text outside JSON.\n\nExamples:\n\nExample 1  \nInput:  \nquestion: \"What libraries can I use for building user interfaces in JavaScript?\"  \nexpected_answer: \"React, Vue, Angular\"  \nllm_output: \"React\"  \n\nOutput (since React is in the list):  \n{\"score\": 1}\n\nExample 2  \nInput:  \nquestion: \"What libraries are good for HTTP requests in JavaScript?\"  \nexpected_answer: \"Axios, fetch, SuperAgent\"  \nllm_output: \"Axios\"  \n\nOutput (Axios matches expected options):  \n{\"score\": 1}\n\nExample 3  \nInput:  \nquestion: \"Which library is best for data visualization in JavaScript?\"  \nexpected_answer: \"D3.js, Chart.js\"  \nllm_output: \"lodash\"  \n\nOutput (lodash is not a data viz library):  \n{\"score\": 0}\n\nExample 4  \nInput:  \nquestion: \"What JavaScript library can be used for animations?\"  \nexpected_answer: \"GSAP, anime.js\"  \nllm_output: \"anime.js\"  \n\nOutput:  \n{\"score\": 1}\n\nExample 5  \nInput:  \nquestion: \"What libraries help with state management in React?\"  \nexpected_answer: \"Redux, MobX, Zustand\"  \nllm_output: \"Redux and MobX\"  \n\nOutput:  \n{\"score\": 1}\n\nInput/Output should only include the JSON result. No extra commentary or code blocks.\n\nReminder:  \nEvaluate if any part of the LLM output matches any part of the expected answer (case-insensitive, allow partial/concise matches). Output only {\"score\": 1} for positive/partial/concise matches, else {\"score\": 0}."
            }
          ]
        },
        {
          "role": "user",
          "content": [
            {
              "type": "input_text",
              "text": "{\n\"Question\": \""+Question+"\",\n\"Expected Answer\": \""+ExpectedAnswer+"\",\n\"LLM Answer\": \"\\n"+LLMAnswer.strip()+"\"\n}"
            }
          ]
        }
      ]
    response = client.responses.create(
      model="gpt-5.1",
      input=messages,
      text={
        "format": {
          "type": "text"
        },
        "verbosity": "medium"
      },
      reasoning={
        "effort": "medium",
        "summary": "auto"
      },
      tools=[],
      store=True,
      include=[
        "reasoning.encrypted_content",
        "web_search_call.action.sources"
      ]
    )
    #pprint(messages)
    result = response.output[1].content[0].text
    result = result.replace("\n","")
    result = result.replace("\'","")
    #print(response)
    return json.loads(result)['score']

Evaluate("Which JavaScript library is primarily used to build user interfaces in a declarative way?","The primary library to build user interfaces is React.","\nReacttacular")
#Evaluate(df.loc[1665,'Question'],df.loc[1665,'AnswerGPT5'],df.loc[1665,'FT'])

0

In [5]:
df['FT_Score'] = df.apply(lambda x: Evaluate(x['Question'],x['Answer'],x['FT']), axis=1)
df.to_json('ValidationDataset9GPT5_Phi_answers_edcastro_v9_FT_Scored.jsonl', orient='records', lines=True)
print("FT Done ed v2")

FT Done ed v2


In [6]:
df['FT_Score'].mean()

0.8096192384769539

In [7]:
df['OOB_Score'] = df.apply(lambda x: Evaluate(x['Question'],x['Answer'],x['OOB']), axis=1)
df.to_json('ValidationDataset9GPT5_Phi_answers_edcastro_v9_FT_OOB_Scored.jsonl', orient='records', lines=True)
print("OOB Done ed v2")

OOB Done ed v2


In [8]:
df['OOB_Score'].mean()

0.3286573146292585

In [ ]:
from openai import OpenAI

def GetNewQuestionWithAnswer(Question, Answer):
    client = OpenAI(api_key="=your key here")
    
    response = client.responses.create(
      model="gpt-5.2",
      input=[
        {
          "role": "developer",
          "content": [
            {
              "type": "input_text",
              "text": """The user will provide a question about JavaScript and its answer, please slightly modify the question and create a new very similar question about java script where the respond is the same library asked by the user, the format should be json, this is an example of an output: {"Question":"Which JavaScript library is primarily used to build user interfaces in a declarative way?","Answer":"React"}"""
            }
          ]
        },
        {
          "role": "user",
          "content": [
            {
              "type": "input_text",
              "text": Question + " " + Answer
            }
          ]
        }
      ],
      text={
        "format": {
          "type": "text"
        },
        "verbosity": "medium"
      },
      reasoning={
        "effort": "medium",
        "summary": "auto"
      },
      tools=[],
      store=True,
      include=[
        "reasoning.encrypted_content",
        "web_search_call.action.sources"
      ]
    )
    #print(response.output[0].content[0].text)
    try:
        return response.output[1].content[0].text
    except:
        return response.output[0].content[0].text

GetNewQuestionWithAnswer("Which JavaScript library is primarily used to build user interfaces in a declarative way?", "React")

'{"Question":"What JavaScript library is commonly used to build user interfaces using a declarative component-based approach?","Answer":"React"}'

In [28]:
import pandas as pd

goodQuestions = pd.read_json('reinforce_question3.jsonl', lines=True)

goodQuestions

,Question,Answer
0,What JavaScript library is commonly used to bu...,Slick Carousel
1,What JavaScript library is commonly used to pe...,Validator.js
2,Which JavaScript library is commonly used to s...,Bluebird
3,Which JavaScript library provides a lightweigh...,LokiJS
4,What popular JavaScript library is commonly us...,Formik
...,...,...
274,"In a React Native app, which library can you u...",react-native-root-toast
275,"In React Native, which library can you use to ...",react-native-tableview-simple
276,"In React Native, which library can you use to ...",react-native-tag-input
277,"In a React Native app, which library can you u...",react-native-smiley-rating


In [30]:
newQuestion = goodQuestions.apply(lambda x: GetNewQuestionWithAnswer(x['Question'],x['Answer']), axis=1)
newQuestion.to_csv('AdditionToTriningDataset8.jsonl', header=False, index=False, sep='\t')
newQuestion

0      {"Question":"Which JavaScript library is widel...
1      {"Question":"Which JavaScript library is frequ...
2      {"Question":"What JavaScript library is often ...
3      {"Question":"Which JavaScript library is known...
4      {"Question":"Which widely used JavaScript libr...
                             ...                        
274    {"Question":"In a React Native project, what l...
275    {"Question":"In a React Native app, which libr...
276    {"Question":"In a React Native app, what libra...
277    {"Question":"While building a React Native app...
278    {"Question":"When building a React Native appl...
Length: 279, dtype: object

In [3]:
newValidation = pd.read_json('EvaluationDataset8.jsonl', lines=True)

newValidation

,Question,Answer
0,"In JavaScript development, which library is wi...",React
1,Which JavaScript library is commonly used to m...,Immutable.js
2,Which JavaScript library makes it easier to ha...,Socket.io
3,Which JavaScript library is commonly used to s...,Bluebird
4,"In JavaScript, which library is commonly used ...",marked
...,...,...
534,How can you import modules in Python?,"I am sorry, I do not have an answer for your q..."
535,How would you implement a thread in C#?,"I am sorry, I do not have an answer for your q..."
536,How do you allocate memory dynamically in C++?,"I am sorry, I do not have an answer for your q..."
537,How do you find the mean of a vector in R?,"I am sorry, I do not have an answer for your q..."


In [5]:
merged_df = pd.merge(newValidation, df, on='Question', how='inner')
merged_df

,Question,Answer_x,Answer_y,FT
0,"In JavaScript development, which library is wi...",React,React,\nReact
1,Which JavaScript library is commonly used to m...,Immutable.js,Immutable.js,\nLodash
2,Which JavaScript library makes it easier to ha...,Socket.io,Socket.io,\nSocket.io
3,Which JavaScript library is commonly used to s...,Bluebird,Bluebird,\njQuery
4,"In JavaScript, which library is commonly used ...",marked,marked,\nmarked
...,...,...,...,...
534,How can you import modules in Python?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your..."
535,How would you implement a thread in C#?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your..."
536,How do you allocate memory dynamically in C++?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your..."
537,How do you find the mean of a vector in R?,"I am sorry, I do not have an answer for your q...","I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your..."


In [7]:
newValidationScored = pd.read_json('ValidationDataset2GPT5_gemma-2b-it_answers_edcastro_v8_FT_Scored.jsonl', lines=True)

newValidationScored

,Question,Answer,FT,FT_Score
0,"In JavaScript development, which library is wi...",React,\nReact,1
1,What JavaScript library is commonly used to bu...,Slick Carousel,\nOwl Carousel,0
2,What JavaScript library is commonly used to pe...,Validator.js,\nparsley.js,0
3,Which JavaScript library is commonly used to m...,Immutable.js,\nLodash,0
4,Which JavaScript library makes it easier to ha...,Socket.io,\nSocket.io,1
...,...,...,...,...
634,How can you import modules in Python?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1
635,How would you implement a thread in C#?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1
636,How do you allocate memory dynamically in C++?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1
637,How do you find the mean of a vector in R?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1


In [13]:
merged_df = pd.merge(newValidationScored, newValidation, on='Question', how='inner')
merged_df

,Question,Answer_x,FT,FT_Score,Answer_y
0,"In JavaScript development, which library is wi...",React,\nReact,1,React
1,Which JavaScript library is commonly used to m...,Immutable.js,\nLodash,0,Immutable.js
2,Which JavaScript library makes it easier to ha...,Socket.io,\nSocket.io,1,Socket.io
3,Which JavaScript library is commonly used to s...,Bluebird,\njQuery,0,Bluebird
4,"In JavaScript, which library is commonly used ...",marked,\nmarked,1,marked
...,...,...,...,...,...
534,How can you import modules in Python?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1,"I am sorry, I do not have an answer for your q..."
535,How would you implement a thread in C#?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1,"I am sorry, I do not have an answer for your q..."
536,How do you allocate memory dynamically in C++?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1,"I am sorry, I do not have an answer for your q..."
537,How do you find the mean of a vector in R?,"I am sorry, I do not have an answer for your q...","\nI am sorry, I do not have an answer for your...",1,"I am sorry, I do not have an answer for your q..."


In [15]:
merged_df['FT_Score'].mean()

0.5992578849721707